<a href="https://colab.research.google.com/github/Iffraah96/ShopSense-AI/blob/main/notebook/07_AI_Review_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ShopSense AI

## Notebook 7 — ShopSense AI Prediction System.

### Objective

The workflow will be:

Customer Review

↓

TF-IDF Vectorizer

↓

Logistic Regression Model

↓

Prediction + Confidence

↓

Recommended / Not Recommended

We'll eventually make it interactive so you can type any customer review into the notebook.

## Import Libraries

In [2]:
# General Libraries
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

## Load the Final ShopSense Model

In [3]:
# Load the final Logistic Regression model
model = joblib.load("shopsense_final_model.pkl")

# Load the TF-IDF vectorizer
tfidf = joblib.load("shopsense_final_tfidf.pkl")

print("ShopSense AI model loaded successfully!")
print("TF-IDF vectorizer loaded successfully!")

# Confirm the number of features
print("Number of TF-IDF features:", len(tfidf.get_feature_names_out()))

ShopSense AI model loaded successfully!
TF-IDF vectorizer loaded successfully!
Number of TF-IDF features: 5000


## Create the ShopSense Prediction Function

This function will take a customer's review and return:

Prediction → Recommended / Not Recommended
Confidence → model's estimated probability

In [17]:
# ============================================
# ShopSense AI - Prediction Preprocessing
# Uses the same preprocessing pipeline as Notebook 3
# ============================================

import re
import nltk

# Download the same NLTK resources used during training
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Create the stopword list
stop_words = set(stopwords.words("english"))

# Preserve important negation words
important_words = {
    "not", "no", "never", "neither", "nor",
    "it's", "isn't", "don't", "doesn't",
    "didn't", "can't", "couldn't", "won't",
    "wouldn't", "wasn't", "weren't",
    "shouldn't", "hasn't", "haven't", "hadn't"
}

# Remove important negation words from the stopword list
stop_words = stop_words - important_words

# Create the lemmatizer
lemmatizer = WordNetLemmatizer()


# Step 1: Basic text cleaning
def clean_new_review(text):
    # Convert to lowercase
    text = text.lower()

    # Keep letters, numbers, spaces, and apostrophes
    text = re.sub(r"[^a-z0-9\s']", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Step 2: Apply the same preprocessing used during training
def preprocess_new_review(text):

    # Basic cleaning
    cleaned_text = clean_new_review(text)

    # Tokenization
    tokens = cleaned_text.split()

    # Remove stopwords while preserving negation words
    tokens_without_stopwords = [
        word for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    lemmatized = [
        lemmatizer.lemmatize(word)
        for word in tokens_without_stopwords
    ]

    # Join the words back into one cleaned review
    final_cleaned_review = " ".join(lemmatized)

    return final_cleaned_review


# Create the ShopSense AI prediction function

def predict_review(review):

    # Preprocess the new customer review
    processed_review = preprocess_new_review(review)

    # Convert the processed review into TF-IDF features
    review_tfidf = tfidf.transform([processed_review])

    # Make the prediction
    prediction = model.predict(review_tfidf)[0]

    # Get probabilities for both classes
    probabilities = model.predict_proba(review_tfidf)[0]

    # Get the confidence of the predicted class
    confidence = probabilities[prediction]

    # Convert the numeric prediction into a readable result
    if prediction == 1:
        result = "Recommended"
    else:
        result = "Not Recommended"

    return result, confidence

print("ShopSense prediction pipeline created successfully!")

ShopSense prediction pipeline created successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Test the Prediction System

Let's test it with a new positive customer review.

In [18]:
# Example customer review
review = "I love this dress! The fabric is soft, comfortable, and the fit is perfect."

# Get the prediction
prediction, confidence = predict_review(review)

# Display the result
print("Customer Review:")
print(review)

print("\nShopSense AI Result:")
print(prediction)

print(f"Confidence: {confidence:.2%}")

Customer Review:
I love this dress! The fabric is soft, comfortable, and the fit is perfect.

ShopSense AI Result:
Recommended
Confidence: 99.47%


## Test a Negative Review

Now let's make sure the system also handles a negative review:

In [19]:
# Example negative customer review
review = "I am very disappointed. The material feels cheap and the dress is uncomfortable. I returned it."

# Get the prediction
prediction, confidence = predict_review(review)

# Display the result
print("Customer Review:")
print(review)

print("\nShopSense AI Result:")
print(prediction)

print(f"Confidence: {confidence:.2%}")

Customer Review:
I am very disappointed. The material feels cheap and the dress is uncomfortable. I returned it.

ShopSense AI Result:
Not Recommended
Confidence: 97.86%


In [28]:
# Test multiple customer reviews

test_reviews = [
    "I absolutely love this dress. The fabric is soft and the fit is perfect.",
    "I was disappointed. The material feels cheap and the fit is terrible.",
    "The dress is beautiful but the fit is uncomfortable.",
    "I don't like this product at all.",
    "The color is beautiful and the fabric feels amazing."
]

# Run ShopSense on each review
for review in test_reviews:

    prediction, confidence = predict_review(review)

    print("=" * 60)
    print("Review:", review)
    print("Prediction:", prediction)
    print(f"Confidence: {confidence:.2%}")

Review: I absolutely love this dress. The fabric is soft and the fit is perfect.
Prediction: Recommended
Confidence: 99.31%
Review: I was disappointed. The material feels cheap and the fit is terrible.
Prediction: Not Recommended
Confidence: 94.81%
Review: The dress is beautiful but the fit is uncomfortable.
Prediction: Recommended
Confidence: 75.94%
Review: I don't like this product at all.
Prediction: Recommended
Confidence: 71.80%
Review: The color is beautiful and the fabric feels amazing.
Prediction: Recommended
Confidence: 90.07%


## Interactive Customer Review Prediction

In [30]:
# ============================================
# ShopSense AI - Interactive Prediction
# ============================================

review = input("Enter a customer review: ")

prediction, confidence = predict_review(review)

print("\n" + "=" * 50)
print("             ShopSense AI")
print("=" * 50)

print("\nCustomer Review:")
print(review)

print("\nPrediction:")
print(prediction)

print(f"\nConfidence: {confidence:.2%}")

print("=" * 50)

Enter a customer review: THE PRODUCT IS AWFUL

             ShopSense AI

Customer Review:
THE PRODUCT IS AWFUL

Prediction:
Not Recommended

Confidence: 82.95%


## Concluding Observations

Notebook 7 converted the trained ShopSense AI Logistic Regression model into a simple prediction system that can analyze new customer reviews.

The prediction pipeline applies the same text preprocessing used during model training, including cleaning, tokenization, stopword removal while preserving important negation words, and lemmatization. The processed review is then transformed using the saved TF-IDF vectorizer and classified using the trained Logistic Regression model.

The system successfully produces a recommendation prediction and confidence score for new customer reviews.

Testing also revealed an important limitation of the baseline model. Although the model performed strongly during evaluation, it can struggle with contextual language and negation. For example, phrases such as "don't like it at all" may be incorrectly classified because TF-IDF represents text primarily through learned word and phrase features rather than understanding the complete meaning of a sentence.

This demonstrates that ShopSense AI provides a strong baseline recommendation classifier while also identifying opportunities for future improvement through models with stronger contextual language understanding.